(quick-start-ml)=
# Quick start tutorial for machine learning



This notebook provides a quick overview of developing serverless functions to train and deploy models using the [MLRun](https://www.mlrun.org/) AI orchestration framework.

**In this tutorial**
- [**Install MLRun**](#install-mlrun)
- [**Define the MLRun project and ML functions**](#define-mlrun-project-and-ml-functions)
- [**Run the data processing function and log artifacts**](#run-your-data-processing-function-and-log-artifacts)
- [**Use the MLRun built-in Function Hub functions for training**](#train-a-model-using-an-mlrun-built-in-function-hub)
- [**Build, test, and deploy model serving functions**](#build-test-and-deploy-the-model-serving-functions)

<iframe width="560" height="315" src="https://www.youtube.com/embed/xI8KVGLlj7Q" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe>

<a id="install"></a>
## Install MLRun

MLRun has a backend service that can run locally or over Kubernetes (preferred). See the instructions for installing it [over Kubernetes Cluster](https://docs.mlrun.org/en/stable/install/kubernetes.html). Alternatively, you can use Iguazio's [managed MLRun service](https://www.iguazio.com/docs/latest-release/).

**Before you start, make sure the MLRun client package is installed and configured properly:**

This notebook uses `sklearn`. If it is not installed in your environment, run `!pip install scikit-learn~=1.5.2`.

In [ ]:
# Install MLRun and sklearn, run this only once (restart the notebook after the install !!!)
%pip install mlrun scikit-learn~=1.5.2

**Restart the notebook kernel after the pip installation.**

In [ ]:
import mlrun

<a id="set-env"></a>
### Configure the client environment

MLRun client connects to the local or remote MLRun service/cluster using a REST API. To configure the service address, credentials, and default settings, you use the `mlrun.set_environment()` method, or environment variables, (see details in [Set up your client environment](https://docs.mlrun.org/en/stable/install/remote.html).)

You can skip this step when using MLRun Jupyter notebooks or Iguazio's managed notebooks.

<a id="define-project"></a>
## Define MLRun project and ML functions

[MLRun **Project**](https://docs.mlrun.org/en/stable/projects/project.html) is a container for all your work on a particular activity or application. Projects host `functions`, `workflow`,
`artifacts`, `secrets`, and more. Projects have access control and can be accessed by one or more users. They are usually associated with a GIT and interact with CI/CD frameworks for automation.
See the MLRun [Projects documentation](https://docs.mlrun.org/en/stable/projects/project.html).

**Create a new project**

In [ ]:
project = mlrun.get_or_create_project("tutorial", "./", user_project=True)

[MLRun serverless functions](https://docs.mlrun.org/en/stable/runtimes/functions.html) specify the source code, base `image`, extra package `requirements`, runtime engine `kind` (batch `job`, real-time `serving`, `spark`, `dask`, etc.), and desired resources (cpu, gpu, mem, storage, ..). The runtime engines (local, job, Nuclio, Spark, etc.) automatically transform the function code and spec into fully managed and elastic services that run over Kubernetes.
Function source code can come from a single file (.py, .ipynb, etc.) or a full archive (git, zip, tar). MLRun can execute an entire file/notebook or specific function classes/handlers.

```{admonition} Note
The `@mlrun.handler` is a decorator that logs the returning values to MLRun as configured. This example uses the default settings so that it logs a dataset (`pd.DataFrame`) and a string value by getting the returned objects types. In addition to logging outputs, the decorator can parse incoming inputs to the required type. For more info, see the [`mlrun.handler`](https://docs.mlrun.org/en/stable/api/mlrun.html#mlrun.handler) documentation.
```

**Function code**

Run the following cell to generate the data prep file (or copy it manually):

In [ ]:
%%writefile src/data-prep.py

import pandas as pd
from sklearn.datasets import load_breast_cancer


def breast_cancer_generator():
    """
    A function which generates the breast cancer dataset
    """
    breast_cancer = load_breast_cancer()
    breast_cancer_dataset = pd.DataFrame(
        data=breast_cancer.data, columns=breast_cancer.feature_names
    )
    breast_cancer_labels = pd.DataFrame(data=breast_cancer.target, columns=["label"])
    breast_cancer_dataset = pd.concat(
        [breast_cancer_dataset, breast_cancer_labels], axis=1
    )

    return breast_cancer_dataset, "label"

**Create a serverless function object from the code above, and register it in the project**

In [ ]:
data_gen_fn = project.set_function(
    "src/data-prep.py",
    name="data-prep",
    kind="job",
    image="mlrun/mlrun",
    handler="breast_cancer_generator",
)
project.save()  # save the project with the latest config

<a id="run-function"></a>
## Run your data processing function and log artifacts
Functions are executed (using the CLI or SDK **`run`** command) with an optional `handler`, various `params`, `inputs`, and resource requirements. This generates a `run` object that can be tracked through the CLI, UI, and SDK. Multiple functions can be executed and tracked as part of a multi-stage pipeline (workflow). 

```{admonition} Note
When a function has additional package `requirements`, or needs to include the content of a `source` archive,
you must first build the function using the `project.build_function()` method.
```

The `local` flag indicates if the function is executed **locally** or "teleported" and executed in the **Kubernetes cluster**. The execution progress and results can be viewed in the UI (see hyperlinks below).

<br>

**Run using the SDK**

In [ ]:
gen_data_run = project.run_function(
    "data-prep", local=True, returns=["dataset", "label_column"]
)

<br>

**Print the run state and outputs**

In [ ]:
gen_data_run.state()

In [ ]:
gen_data_run.outputs

<br>

**Print the output dataset artifact (`DataItem` object) as dataframe**

In [ ]:
gen_data_run.artifact("dataset").as_df().head()

<a id="use-hub"></a>
## Train a model using an MLRun built-in Function Hub

MLRun provides a [**Function Hub**](https://www.mlrun.org/hub/) that hosts a set of pre-implemented and
validated ML, DL, and data processing functions.

You can import the `auto-trainer` hub function that can: train an ML model using a variety of ML frameworks; generate
various metrics and charts; and log the model along with its metadata into the MLRun model registry.

In [ ]:
# Import the function
trainer = mlrun.import_function("hub://auto_trainer")


See the `auto_trainer` function usage instructions in [the Function Hub](https://www.mlrun.org/hub/functions/master/auto_trainer/) or by typing `trainer.doc()`

**Run the function on the cluster (if there is)**

In [ ]:
trainer_run = project.run_function(
    trainer,
    inputs={"dataset": gen_data_run.outputs["dataset"]},
    params={
        "model_class": "sklearn.ensemble.RandomForestClassifier",
        "train_test_split_size": 0.2,
        "label_columns": "label",
        "model_name": "cancer",
    },
    handler="train",
)

<br>

**View the job progress results and the selected run in the MLRun UI**

![train job in UI](./_static/images/train-job.png)


<br>

**Results (metrics) and artifacts are generated and tracked automatically by MLRun**

In [ ]:
trainer_run.outputs

In [ ]:
# Display HTML output artifacts
trainer_run.artifact("confusion-matrix").show()

<a id="model-serving"></a>
## Build, test, and deploy the model serving functions

MLRun serving can produce managed, real-time, serverless, pipelines composed of various data processing and ML tasks. The pipelines use the Nuclio real-time serverless engine, which can be deployed anywhere. For more details and examples, see [MLRun serving graphs](https://docs.mlrun.org/en/stable/serving/serving-graph.html).

**Create a model serving function**

In [ ]:
serving_fn = project.set_function(
    func="",
    name="serving",
    image="mlrun/mlrun",
    kind="serving",
    requirements=["scikit-learn~=1.5.2"],
)

**Add a model**

The basic serving topology supports a router with multiple child models attached to it.
The `function.add_model()` method allows you to add models and specify the `name`, `model_path` (to a model file, dir, or artifact), and the serving `class` (built-in or user defined).

In [ ]:
serving_fn.add_model(
    "cancer-classifier",
    model_path=trainer_run.outputs["model"],
    class_name="mlrun.frameworks.sklearn.SKLearnModelServer",
)

In [ ]:
# Plot the serving graph topology
serving_fn.spec.graph.plot(rankdir="LR")

**Simulating the model server locally**

In [ ]:
# Create a mock (simulator of the real-time function)
server = serving_fn.to_mock_server()

<br>

**Test the mock model server endpoint**
    
- List the served models

In [ ]:
server.test("/v2/models/", method="GET")

- Infer using test data

In [ ]:
my_data = {
    "inputs": [
        [
            1.371e01,
            2.083e01,
            9.020e01,
            5.779e02,
            1.189e-01,
            1.645e-01,
            9.366e-02,
            5.985e-02,
            2.196e-01,
            7.451e-02,
            5.835e-01,
            1.377e00,
            3.856e00,
            5.096e01,
            8.805e-03,
            3.029e-02,
            2.488e-02,
            1.448e-02,
            1.486e-02,
            5.412e-03,
            1.706e01,
            2.814e01,
            1.106e02,
            8.970e02,
            1.654e-01,
            3.682e-01,
            2.678e-01,
            1.556e-01,
            3.196e-01,
            1.151e-01,
        ]
    ]
}
server.test("/v2/models/cancer-classifier/infer", body=my_data)

- Read the model name, ver and schema (input and output features)

**Deploy a real-time serving function (over Kubernetes or Docker)**

This section requires Nuclio to be installed (over k8s or Docker).

Use the mlrun `deploy_function()` method to build and deploy a Nuclio serving function from your serving-function code.
You can deploy the function object (`serving_fn`) or reference pre-registered project functions.

In [ ]:
project.deploy_function(serving_fn)

- Test the live endpoint

In [ ]:
serving_fn.invoke("/v2/models/cancer-classifier/infer", body=my_data)

## Done!

Congratulations! You've completed Part 1 of the MLRun getting-started tutorial.
Proceed to [**Part 2: Train, compare, and register models**](02-model-training.ipynb) to learn how to train an ML model.